# ForceLorentzBalance — coupled low-Rm MHD `force_lorentz`

`MagneticCoupled` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **Lorentz-force coupling** between the magnetic induction
equation and the momentum equation of the same `Pb_MHD` problem. The problem
block contains **two equations**: `Convection_Diffusion_Magnetic_field` and the
optional `Navier_Stokes_standard`, coupled by the `force_lorentz` source term
attached to the momentum equation:

$$\mathbf{F} = \mathbf{J}\times\mathbf{b},\qquad \mathbf{J} = \frac{1}{\mu_0}\nabla\times\mathbf{b}$$

computed from the **lagged** magnetic field (explicit low-Rm coupling) and its
Green-Gauss element gradient. The source is expressed per unit mass, like every
TRUST momentum volumetric source (gravity/Boussinesq convention), and is
assembled on the face-based velocity unknown following the kernel
`Terme_Boussinesq_PolyMAC_CDO_Face` structure (face block + PolyMAC_MPFA
element block, Dirichlet-velocity faces skipped).

This notebook runs **one and only one test case**: `equilibre_lorentz`.

## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | Lorentz hydrostatic balance | $p(y) = -y^2/(2\mu_0^2\rho) + C$ |
| 2 | curl-free force produces no flow | $\mathbf{u} \to 0$ |

With the exactly linear field $\mathbf{b} = (y, 0)$ ($\eta = 0$, no transport
velocity, so $\mathbf{b}$ is an exact equilibrium):

$$\nabla\times\mathbf{b} = -1 \;\Rightarrow\; \mathbf{J} = (0,0,-1)/\mu_0 \;\Rightarrow\; \mathbf{F} = \mathbf{J}\times\mathbf{b} = (0,-y,0)/\mu_0^2$$

$\mathbf{F}$ is a pure gradient, so the exact steady solution of the momentum
equation in the closed box is $\mathbf{u} = 0$ and
$p = -y^2/(2\mu_0^2\rho) + C$.  With $\mu_0 = \rho = 1$, the pressure
difference between the interior cell centres $(0.35, 0.45)$ and $(0.35, 0.75)$
is $p_2 - p_1 = -(0.75^2 - 0.45^2)/2 = -0.18$.

**Tolerances.** The Green-Gauss $\nabla\mathbf{b}$ is exact at interior cells
but carries the O(h) wall bias (homogeneous-Neumann boundary assumption), and
the discrete force balances the discrete pressure gradient only projectively:
the pressure difference converges to $-0.1853$ instead of $-0.18$ (~3 %, mesh
convergent) and the velocity transient decays exponentially without reaching
machine zero. The assertions below bound both effects.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/equilibre_lorentz.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NX0 = NY0 = 11                        # baseline node counts (coarse)
meshes = {
    "coarse":   {"nx": 11, "ny": 11},            # the baseline mesh
    "standard": {"nx": 2*NX0-1, "ny": 2*NY0-1},
    "fine":     {"nx": 4*NX0-3, "ny": 4*NY0-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("equilibre_lorentz.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition equilibre_lorentz.data 2 -> PAR_equilibre_lorentz.data + DOM_000*.Zones
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks

### #1 — Lorentz hydrostatic balance (pressure difference)



In [ ]:
t1, p1 = read_son(case.dataFileName_, "SONDE_P1")
t2, p2 = read_son(case.dataFileName_, "SONDE_P2")
assert np.allclose(t1, t2), "probe time grids differ"

dp = p2 - p1
expected = -(0.75**2 - 0.45**2) / 2.0        # -0.18, with mu0 = rho = 1
print(f"p(0.45) = {p1[-1]:.6g}")
print(f"p(0.75) = {p2[-1]:.6g}")
print(f"p2 - p1 = {dp[-1]:.6g}   (analytic {expected:.4g}, "
      f"discrete balance ~ -0.1853 at h = 0.1)")
err = abs(dp[-1] - expected) / abs(expected)
print(f"relative error = {err:.3e}")
assert dp[-1] < 0, "wrong sign of the Lorentz pressure balance"
assert err < 0.10, f"pressure balance off: dp = {dp[-1]} (expected {expected} +/- 10 %)"
print("OK: Lorentz hydrostatic balance (sign and magnitude within 10 %).")



### #2 — curl-free force produces no flow



In [ ]:
def read_son_all(case, sonde):
    """Every entry (one per point/component) of probe `sonde` as (name, values)."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

entries = read_son_all(case.dataFileName_, "SONDE_U")
# The transient peaks at the first step (~3e-3) then decays like
# exp(-nu pi^2 t); the case runs to tmax = 0.5 (10 steps).
# Check the relaxed state: |u| at the final time below 5e-4.
unorm = max(float(np.max(np.abs(v[-1:]))) for _, v in entries)
print(f"max |u| over all probe entries (final time): {unorm:.3e}")
for name, v in entries:
    print(f"  {name}: last |u| = {np.max(np.abs(v[-1:])):.3e}")
assert unorm < 5e-4, f"velocity did not relax: |u| = {unorm}"
print("OK: the curl-free Lorentz force is balanced by pressure, u -> 0.")



## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`11×11`),
`standard` = cells ×2 (`21×21`), `fine` = cells ×4 (`41×41`).

Metric: the **Lorentz hydrostatic balance** error
$e_h = |\Delta p_h - \Delta p_{\mathrm{exact}}|$, where the exact pressure
drop $p = -(x^2+y^2)/\mu_0^2\rho$ is evaluated at each probe's
**effective** (cell-centre) position read from the `.son` header — under
refinement the probes snap to different cell centres, so the nominal
comparison would carry an $O(h)$ position error that is not a physics
error. The Green–Gauss $\mathrm{curl}\,b$ reconstruction is $O(h^2)$ at
the interior probes, and the observed order is $\log_2(e_k/e_{k+1})$.

In [ ]:
def probe_pos(path):
    """Effective (x, y) of a point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return (float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2]))
    raise RuntimeError(f"no probe position header in {path}")

def read_son_dir(subdir, case_name, name):
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float), probe_pos(path)

def dp_error(subdir, case_name):
    """(abs err, exact drop) of the Lorentz hydrostatic pressure balance for
    b = (y, 0): J x B = -(0, y, 0)/mu0^2 -> p = y^2/2 + C, evaluated at the
    probes' effective cell-centre positions (mu0 = rho = 1)."""
    _, p1, xy1 = read_son_dir(subdir, case_name, "SONDE_P1")
    _, p2, xy2 = read_son_dir(subdir, case_name, "SONDE_P2")
    dp_h = float(p2[-1] - p1[-1])
    dp_exact = -(xy2[1]**2 - xy1[1]**2) / 2.0
    return abs(dp_h - dp_exact), dp_exact

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (11, 11), "standard": (21, 21), "fine": (41, 41)}
res = {lvl: dp_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes      |dp - dp_exact(eff)|   rel      observed order")
prev = None
for lvl in ("coarse", "standard", "fine"):
    nx, ny = scan_nodes[lvl]
    e, dp_x = res[lvl]
    rel = e / abs(dp_x)
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<9} {nx:>3}x{ny:<4} {e:<22.4e} {rel:<8.3f} {order}")
    prev = e
    # The baseline contract (dp within 5 pct of the exact drop) must hold at
    # EVERY level: the residual has a mesh-independent O(dt) floor (the
    # balance is reached only to temporal accuracy), so a strict monotone
    # decrease is not structural here.
    assert rel < 0.05, f"hydrostatic balance off at {lvl}: rel {rel:.3f}"
print("OK: hydrostatic Lorentz balance holds within 5 pct at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()

